# YOLO vs. DETR — Zorlu Koşullarda Hız–Doğruluk (mAP–FPS) Ödünleşimi

Bu notebook, tek aşamalı bir CNN dedektörü (**YOLOv8**) ile transformer tabanlı uçtan uca bir dedektörü (**RT-DETR**, DETR ailesi) **aynı veri, aynı ayrım ve aynı değerlendirme protokolü** altında karşılaştırır.

| Bölüm | İçerik |
|---|---|
| 1 | Kurulum, yapılandırma ve tekrarlanabilirlik |
| 2 | **Sızıntısız train/val/test ayrımı** (birebir + yakın kopya + sahne gruplama, denetim testleri) |
| 3 | Zorlu koşul simülasyonu: sis, düşük ışık, hareket bulanıklığı, Gauss gürültüsü, yağmur |
| 4 | **Baseline ve ablasyon senaryoları** |
| 5 | Eğitim + değerlendirme döngüsü (**3 tohum**), mAP ve FPS ölçümü |
| 6 | **Ortalama ± standart sapma**, %95 GA, eşleştirilmiş t-testleri (Holm düzeltmeli) |
| 7 | **Matplotlib grafikleri**: kayıp/mAP eğrileri, mAP–FPS Pareto, dayanıklılık |
| 8 | Yorumlama rehberi ve sınırlılıklar |

**Neden RT-DETR?** Orijinal DETR (Carion vd., 2020) yakınsamak için ~300–500 epoch ister; bir ders projesinde bu pratik değildir. RT-DETR aynı ailenin (Hungarian eşleme, NMS'siz kümesel tahmin) gerçek zamanlı sürümüdür ve Ultralytics'te YOLO ile **birebir aynı eğitim/değerlendirme hattını** paylaşır; bu, karşılaştırmada çatı (framework) farkından kaynaklı yanlılığı ortadan kaldırır.

**Çalıştırma profili (`PROFILE`):** `"quick"` tüm hattı birkaç epoch ile uçtan uca doğrular (sonuçlar anlamlı değildir). `"mid"` rapor için önerilen ayardır (30 epoch, verinin tamamı, 3 senaryo × 3 tohum). `"full"` tam deneydir (80 epoch, 4 senaryo). Colab'da sonuçlar Google Drive'a yazılır; oturum koparsa notebook'u yeniden çalıştırın, biten çalıştırmalar atlanır.

In [ ]:
# 0) Kurulum — gerekirse bir kez çalıştırın
%pip install -q "ultralytics>=8.1.0" scipy pandas matplotlib

## 1. Yapılandırma ve tekrarlanabilirlik

* `split_seed` **veri ayrımını** sabitler; eğitim tohumlarından (`seeds`) **bağımsızdır**. Böylece 3 tohumun hepsi *aynı* test kümesinde değerlendirilir ve tohumlar arası fark yalnızca eğitim stokastikliğini (ağırlık başlatma, veri sırası, artırma) yansıtır.
* Modeller kapasite olarak yakın seçilmiştir (YOLOv8m ≈ 26M, RT-DETR-L ≈ 32M parametre). Parametre sayısı sonuç tablosuna ayrıca yazılır.
* **Ön-eğitim sızıntısı uyarısı:** COCO ön-eğitimli ağırlıklarla COCO'dan türetilmiş bir veri kümesinde (ör. `coco128`) test yapmak, test görüntülerinin ön-eğitimde görülmüş olması demektir. Bu yüzden varsayılan veri kümesi COCO dışı bir kümedir (*African Wildlife*, 4 sınıf, ~1500 görüntü).

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # deterministik cuBLAS; torch'tan ÖNCE ayarlanmalı

import re, gc, json, time, random, shutil, hashlib, platform, zipfile, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image
from scipy import stats

import ultralytics
from ultralytics import YOLO, RTDETR

warnings.filterwarnings("ignore")

# Çalıştırma profili
#   "quick": duman testi — 3 epoch, verinin %25'i, 2 senaryo (sonuçlar anlamlı değildir)
#   "mid"  : rapor için önerilen — 30 epoch, verinin tamamı, 3 senaryo (2 × 3 × 3 = 18 çalıştırma)
#   "full" : tam deney — 80 epoch, verinin tamamı, 4 senaryo (24 çalıştırma)
PROFILE = "mid"
PROFILES = {
    "quick": {"epochs": 3,  "train_fraction": 0.25, "fps_iters": 30,
              "scenarios": ["baseline", "abl_adv_aug"]},
    "mid":   {"epochs": 30, "train_fraction": 1.0,  "fps_iters": 200,
              "scenarios": ["baseline", "abl_adv_aug", "abl_imgsz480"]},
    "full":  {"epochs": 80, "train_fraction": 1.0,  "fps_iters": 200,
              "scenarios": ["baseline", "abl_adv_aug", "abl_imgsz480", "abl_no_mosaic"]},
}
P = PROFILES[PROFILE]
QUICK_MODE = PROFILE == "quick"

# Colab'da SONUÇLAR Google Drive'a yazılır: bağlantı koparsa biten çalıştırmalar kaybolmaz ve
# yeniden çalıştırıldığında atlanır. VERİ yerel diskte kalır (Drive'dan okumak eğitimi çok yavaşlatır);
# ayrım deterministik olduğu için her oturumda birebir aynı ayrım yeniden üretilir.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/cv_bench")
    DATA_ROOT = Path("/content/cv_bench_data")
except ImportError:
    ROOT = Path("cv_bench").resolve()
    DATA_ROOT = ROOT / "data"

CFG = {
    # --- veri ---
    "data_root": DATA_ROOT,
    "runs_root": ROOT / "runs",
    "fig_dir": ROOT / "figures",
    "dataset_url": "https://github.com/ultralytics/assets/releases/download/v0.0.0/african-wildlife.zip",
    "source_dir": None,            # Kendi YOLO formatlı veriniz için: Path("...") (içinde .../images/... ve .../labels/... olmalı)
    "class_names": ["buffalo", "elephant", "rhino", "zebra"],
    "group_regex": None,           # Video/sahne kareleri için ör. r"^(.*)_frame\d+$" -> aynı videonun kareleri aynı bölmeye düşer
    "split_seed": 2024,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "dhash_hamming_thr": 5,        # 64 bitlik dHash'te <=5 bit fark = yakın kopya
    # --- deney ---
    "seeds": [0, 1, 2],
    "models": {                    # ad: (aile, ön-eğitimli ağırlık)
        "YOLOv8m":   ("yolo",   "yolov8m.pt"),
        "RT-DETR-L": ("rtdetr", "rtdetr-l.pt"),
    },
    "epochs": P["epochs"],
    "batch": 8,
    "train_fraction": P["train_fraction"],
    "workers": 2,
    "device": 0 if torch.cuda.is_available() else "cpu",
    # --- zorlu koşullar ---
    "test_corruptions": ["fog", "low_light", "motion_blur", "gaussian_noise", "rain"],
    "train_corruptions": ["fog", "low_light", "gaussian_noise"],   # motion_blur ve rain eğitimde HİÇ görülmez (genelleme testi)
    "test_severity": 3,            # 1 (hafif) … 5 (şiddetli)
    "adv_fraction": 0.5,           # ablasyonda eğitim görüntülerinin bozulan oranı (küme boyutu SABİT kalır)
    # --- hız ölçümü ---
    "fps_warmup": 10,
    "fps_iters": P["fps_iters"],
}
for k in ("data_root", "runs_root", "fig_dir"):
    CFG[k].mkdir(parents=True, exist_ok=True)


def set_seed(seed: int):
    """Python, NumPy ve PyTorch (CPU+GPU) tohumlarını sabitler."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False   # FPS'i iki model için de aynı şekilde etkiler -> adil


# Windows'ta Türkçe karakterli yollarla da çalışan okuma/yazma (cv2.imread unicode yolları okuyamaz)
def imread(p):
    return cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_COLOR)

def imwrite(p, img):
    ext = Path(p).suffix.lower()
    params = [cv2.IMWRITE_JPEG_QUALITY, 95] if ext in (".jpg", ".jpeg") else []
    ok, buf = cv2.imencode(ext, img, params)
    assert ok, f"Yazılamadı: {p}"
    buf.tofile(str(p))


env = {
    "python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "PROFILE": PROFILE, "epochs": CFG["epochs"], "train_fraction": CFG["train_fraction"],
}
(ROOT / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
env

## 2. Sızıntısız veri ayrımı

Rastgele görüntü düzeyinde bölme, sızıntıya üç yoldan açıktır. Her biri ayrı bir mekanizmayla engellenir:

1. **Birebir kopyalar** (farklı ad, aynı bayt) → MD5 ile tespit.
2. **Yakın kopyalar** (yeniden boyutlandırılmış/sıkıştırılmış/kırpılmış sürümler) → 64 bitlik *difference hash* (dHash) + Hamming mesafesi.
3. **Aynı sahne/video kareleri** → isteğe bağlı `group_regex`.

Bu ilişkiler **Union-Find** ile gruplara birleştirilir ve bölme **grup düzeyinde** yapılır: bir grup tamamen tek bir bölmeye düşer. Ardından:

* Ayrım `split_manifest.json` içine **dondurulur**; sonraki çalıştırmalar aynı ayrımı yeniden kullanır (test kümesi asla değişmez).
* Denetim (audit) hücresi, bölmeler arasında ortak MD5, ortak grup ve eşik altı dHash mesafesi **olmadığını `assert` ile** doğrular.
* **Kural:** Bozulmalar ve artırmalar her zaman ayrımdan **sonra** ve yalnızca ilgili bölmenin içinde uygulanır; hiperparametre/epoch seçimi yalnızca *val* ile yapılır, *test* yalnızca en sonda bir kez kullanılır.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def download_dataset(url: str, dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    zpath = dest / Path(url).name
    if not zpath.exists():
        print("İndiriliyor:", url)
        urllib.request.urlretrieve(url, zpath)
    marker = dest / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(dest)
        marker.touch()
    return dest


def label_path_for(img: Path) -> Path:
    """.../images/.../x.jpg  ->  .../labels/.../x.txt (son 'images' bileşeni değiştirilir)."""
    parts = list(img.parts)
    i = len(parts) - 1 - parts[::-1].index("images")
    parts[i] = "labels"
    return Path(*parts).with_suffix(".txt")


SRC = Path(CFG["source_dir"]) if CFG["source_dir"] else download_dataset(CFG["dataset_url"], CFG["data_root"] / "raw")
images = sorted(p for p in SRC.rglob("*") if p.suffix.lower() in IMG_EXTS and "images" in p.parts)
n_lbl = sum(label_path_for(p).exists() for p in images)
print(f"Havuzdaki görüntü: {len(images)}  |  etiketli: {n_lbl}  (orijinal train/val/test klasörleri YOK SAYILIR, havuz yeniden bölünür)")

In [ ]:
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def file_md5(p: Path) -> str:
    return hashlib.md5(p.read_bytes()).hexdigest()


def dhash(p: Path, size: int = 8) -> np.ndarray:
    """64 bitlik difference-hash (8 bayt). Yeniden boyutlandırma/JPEG sıkıştırmasına dayanıklıdır."""
    with Image.open(p) as im:
        g = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel())


def min_cross_hamming(Ha: np.ndarray, Hb: np.ndarray, chunk: int = 256) -> int:
    best = 64
    for s in range(0, len(Ha), chunk):
        d = POPCOUNT[Ha[s:s + chunk, None, :] ^ Hb[None, :, :]].sum(-1)
        best = min(best, int(d.min()))
    return best


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def build_groups(paths, thr, group_regex=None):
    n = len(paths)
    md5s = [file_md5(p) for p in paths]
    H = np.stack([dhash(p) for p in paths])
    uf = UnionFind(n)
    # (1) birebir kopyalar
    first = {}
    for i, h in enumerate(md5s):
        if h in first:
            uf.union(i, first[h])
        else:
            first[h] = i
    # (2) yakın kopyalar
    n_pairs = 0
    for s in range(0, n, 256):
        d = POPCOUNT[H[s:s + 256, None, :] ^ H[None, :, :]].sum(-1)
        ii, jj = np.nonzero(d <= thr)
        for i, j in zip(ii + s, jj):
            if i < j:
                uf.union(int(i), int(j))
                n_pairs += 1
    # (3) aynı sahne/video
    if group_regex:
        key_first = {}
        for i, p in enumerate(paths):
            m = re.match(group_regex, p.stem)
            if m:
                k = m.group(1)
                if k in key_first:
                    uf.union(i, key_first[k])
                else:
                    key_first[k] = i
    gid = [uf.find(i) for i in range(n)]
    print(f"Yakın/birebir kopya çifti: {n_pairs}  |  grup sayısı: {len(set(gid))} / {n} görüntü")
    return md5s, H, gid


def group_split(gid, ratios, seed):
    """Grup düzeyinde bölme: büyük gruplar önce, her grup hedefine en uzak bölmeye gider."""
    rng = np.random.default_rng(seed)
    groups = {}
    for i, g in enumerate(gid):
        groups.setdefault(g, []).append(i)
    keys = list(groups)
    keys = [keys[k] for k in rng.permutation(len(keys))]   # eşit boyutlu gruplar arasında rastgelelik
    keys.sort(key=lambda k: -len(groups[k]))                 # kararlı sıralama
    N = len(gid)
    target = {s: r * N for s, r in ratios.items()}
    count = {s: 0 for s in ratios}
    assign = {}
    for k in keys:
        s = max(ratios, key=lambda s: (target[s] - count[s]) / target[s])
        for i in groups[k]:
            assign[i] = s
        count[s] += len(groups[k])
    return assign


CLEAN = CFG["data_root"] / "clean"
MANIFEST = CFG["data_root"] / "split_manifest.json"

if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    print("Dondurulmuş ayrım yüklendi ->", MANIFEST, "(yeniden bölmek için data/clean ve manifest'i silin)")
else:
    md5s, H, gid = build_groups(images, CFG["dhash_hamming_thr"], CFG["group_regex"])
    assign = group_split(gid, CFG["split_ratios"], CFG["split_seed"])
    manifest = {"split_seed": CFG["split_seed"], "dhash_hamming_thr": CFG["dhash_hamming_thr"], "items": []}
    for i, p in enumerate(images):
        s = assign[i]
        new_stem = f"{md5s[i][:10]}_{p.stem}"           # klasörler arası ad çakışmasını önler
        (CLEAN / s / "images").mkdir(parents=True, exist_ok=True)
        (CLEAN / s / "labels").mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, CLEAN / s / "images" / (new_stem + p.suffix.lower()))
        lp = label_path_for(p)
        dst_lbl = CLEAN / s / "labels" / (new_stem + ".txt")
        if lp.exists():
            shutil.copy2(lp, dst_lbl)
        else:
            dst_lbl.write_text("")                      # arka plan görüntüsü
        manifest["items"].append({
            "src": str(p), "split": s, "file": new_stem + p.suffix.lower(),
            "md5": md5s[i], "dhash": H[i].tobytes().hex(), "group": int(gid[i]),
        })
    MANIFEST.write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    print("Ayrım oluşturuldu ve donduruldu ->", MANIFEST)
shutil.copy2(MANIFEST, ROOT / "split_manifest.json")   # ayrımın kalıcı kaydı (Drive'da)

In [ ]:
# --- Sızıntı denetimi: herhangi biri başarısız olursa deney BAŞLAMAZ ---
def leakage_audit(manifest, thr):
    by = {s: [it for it in manifest["items"] if it["split"] == s] for s in ("train", "val", "test")}
    rows = []
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        f = lambda k, s: {it[k] for it in by[s]}
        src_ov = f("src", a) & f("src", b)
        md5_ov = f("md5", a) & f("md5", b)
        grp_ov = f("group", a) & f("group", b)
        Ha = np.stack([np.frombuffer(bytes.fromhex(it["dhash"]), np.uint8) for it in by[a]])
        Hb = np.stack([np.frombuffer(bytes.fromhex(it["dhash"]), np.uint8) for it in by[b]])
        dmin = min_cross_hamming(Ha, Hb)
        rows.append({"çift": f"{a} ↔ {b}", "ortak dosya": len(src_ov), "ortak MD5": len(md5_ov),
                     "ortak grup": len(grp_ov), "min dHash mesafesi": dmin, f"> eşik ({thr})": dmin > thr})
        assert not src_ov and not md5_ov and not grp_ov, f"SIZINTI: {a}↔{b}"
        assert dmin > thr, f"SIZINTI (yakın kopya): {a}↔{b} min mesafe={dmin}"
    return pd.DataFrame(rows)


def class_hist(split):
    cnt = np.zeros(len(CFG["class_names"]), dtype=int)
    for f in (CLEAN / split / "labels").glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                cnt[int(float(line.split()[0]))] += 1
    return cnt


display(leakage_audit(manifest, manifest["dhash_hamming_thr"]))
dist = pd.DataFrame({s: class_hist(s) for s in ("train", "val", "test")}, index=CFG["class_names"])
dist.loc["# görüntü"] = [len(list((CLEAN / s / "images").iterdir())) for s in ("train", "val", "test")]
print("✓ Sızıntı denetimi geçti. Bölmelere göre nesne (kutu) sayıları:")
display(dist)

## 3. Zorlu koşul simülasyonu

Beş fotometrik/optik bozulma, 1–5 şiddet ölçeğiyle (ImageNet-C / Michaelis vd. 2019 tarzı) uygulanır. Bozulmalar **geometriyi değiştirmediği için** etiketler aynen kopyalanır.

* Test bozulmaları, `split_seed`'den türetilen **sabit** bir rastgelelikle bir kez üretilir → tüm modeller ve tohumlar **aynı bozuk test görüntülerini** görür.
* Her görüntünün rastgeleliği `md5(tohum | dosya adı | bozulma)`'dan türetilir → çalıştırma sırasından bağımsız, bit düzeyinde tekrarlanabilir.
* Ablasyonda eğitimde yalnızca `fog`, `low_light`, `gaussian_noise` kullanılır; `motion_blur` ve `rain` **görülmemiş** bozulmalar olarak kalır (grafiklerde † ile işaretli).

In [ ]:
def _rng_for(*keys):
    h = hashlib.md5("|".join(map(str, keys)).encode()).hexdigest()
    return np.random.default_rng(int(h[:8], 16))


def c_fog(img, s, rng):
    h, w = img.shape[:2]
    t = [0.25, 0.35, 0.45, 0.55, 0.65][s - 1]
    n = cv2.resize(rng.random((8, 8)).astype(np.float32), (w, h), interpolation=cv2.INTER_CUBIC)
    n = (n - n.min()) / (np.ptp(n) + 1e-6)                          # düşük frekanslı, düzensiz sis yoğunluğu
    depth = np.linspace(1.0, 0.6, h, dtype=np.float32)[:, None]     # görüntü üstü (uzak) daha sisli
    a = np.clip(t * (0.6 + 0.8 * n) * depth, 0, 0.9)[..., None]
    return np.clip(img.astype(np.float32) * (1 - a) + 225.0 * a, 0, 255).astype(np.uint8)


def c_low_light(img, s, rng):
    gamma = [1.4, 1.8, 2.2, 2.6, 3.0][s - 1]
    gain = [0.75, 0.60, 0.45, 0.35, 0.25][s - 1]
    sig = [0.01, 0.02, 0.03, 0.04, 0.05][s - 1]
    x = (img.astype(np.float32) / 255.0) ** gamma * gain
    x = x + rng.normal(0, sig, x.shape).astype(np.float32) * np.sqrt(x + 1e-3)   # sinyale bağlı (shot) gürültü
    x = x + rng.normal(0, sig / 2, x.shape).astype(np.float32)                    # sensör okuma gürültüsü
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)


def c_motion_blur(img, s, rng):
    k = [5, 9, 13, 17, 23][s - 1]
    ker = np.zeros((k, k), np.float32)
    ker[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1.0)
    ker = cv2.warpAffine(ker, M, (k, k))
    ker /= max(ker.sum(), 1e-6)
    return cv2.filter2D(img, -1, ker)


def c_gaussian_noise(img, s, rng):
    sig = [0.04, 0.06, 0.08, 0.10, 0.13][s - 1] * 255
    return np.clip(img.astype(np.float32) + rng.normal(0, sig, img.shape), 0, 255).astype(np.uint8)


def c_rain(img, s, rng):
    h, w = img.shape[:2]
    n = int([150, 300, 500, 700, 900][s - 1] * h * w / (640 * 640))
    L = [10, 15, 20, 25, 30][s - 1]
    ang = np.deg2rad(rng.uniform(-20, 20))
    dx, dy = int(L * np.sin(ang)), int(L * np.cos(ang))
    layer = np.zeros((h, w), np.float32)
    for x, y in zip(rng.integers(0, w, n), rng.integers(0, h, n)):
        cv2.line(layer, (int(x), int(y)), (int(x) + dx, int(y) + dy), 1.0, 1)
    layer = cv2.GaussianBlur(layer, (3, 3), 0)
    out = img.astype(np.float32) * 0.85 + layer[..., None] * 180      # sahne kararır + parlak çizgiler
    return np.clip(out, 0, 255).astype(np.uint8)


CORRUPTIONS = {"fog": c_fog, "low_light": c_low_light, "motion_blur": c_motion_blur,
               "gaussian_noise": c_gaussian_noise, "rain": c_rain}


def build_corrupted_split(src: Path, dst: Path, corr_pool, seed, fraction=1.0, severity=None) -> Path:
    """src bölmesinin bozulmuş bir kopyasını dst'ye yazar. severity=None -> her görüntü için 1..3 rastgele."""
    if (dst / ".done").exists():
        return dst
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)
    log = {}
    for p in sorted((src / "images").iterdir()):
        rng = _rng_for(seed, p.name, *corr_pool)
        img, tag = imread(p), "clean"
        if rng.random() < fraction:
            name = corr_pool[int(rng.integers(len(corr_pool)))]
            sev = severity if severity else int(rng.integers(1, 4))
            img, tag = CORRUPTIONS[name](img, sev, rng), f"{name}@{sev}"
        imwrite(dst / "images" / p.name, img)
        shutil.copy2(src / "labels" / (p.stem + ".txt"), dst / "labels" / (p.stem + ".txt"))
        log[p.name] = tag
    (dst / ".done").write_text(json.dumps(log, indent=0), encoding="utf-8")
    return dst


# Sabit (dondurulmuş) bozuk test kümeleri — YALNIZCA temiz test bölmesinden türetilir
TEST_DIRS = {"clean": CLEAN / "test"}
for c in CFG["test_corruptions"]:
    TEST_DIRS[c] = build_corrupted_split(CLEAN / "test", CFG["data_root"] / f"test_{c}", [c],
                                         seed=CFG["split_seed"], fraction=1.0, severity=CFG["test_severity"])

clean_test_names = {p.name for p in (CLEAN / "test" / "images").iterdir()}
for c, d in TEST_DIRS.items():
    assert {p.name for p in (d / "images").iterdir()} == clean_test_names, f"{c} test kümesi temiz testle eşleşmiyor"
print("✓ Bozuk test kümeleri hazır:", list(TEST_DIRS))

In [ ]:
# Görsel sağlama: aynı test görüntüsü, tüm koşullar (şiddet = test_severity)
sample = sorted((CLEAN / "test" / "images").iterdir())[0]
fig, axes = plt.subplots(1, len(TEST_DIRS), figsize=(3.2 * len(TEST_DIRS), 3.2))
for ax, (c, d) in zip(axes, TEST_DIRS.items()):
    ax.imshow(cv2.cvtColor(imread(d / "images" / sample.name), cv2.COLOR_BGR2RGB))
    ax.set_title(c, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Birim testleri

Aşağıdaki testler, deney hattının yapı taşlarını **sentetik veriyle** ve veri kümesinden bağımsız olarak sınar. Herhangi bir test başarısız olursa hücre `AssertionError` verir ve eğitime geçilmez. Sonuçlar `unit_tests.csv` dosyasına yazılır (rapordaki Test Süreçleri bölümü için).

| Test | Neyi doğrular |
|---|---|
| `test_label_path_for` | Görüntü → etiket yolu eşlemesi, iç içe `images` klasörlerinde de doğru mu |
| `test_dhash_near_duplicate_and_distinct` | Yeniden boyutlandırılmış + JPEG kopya eşik altında, farklı görüntü eşik üstünde mi |
| `test_union_find_transitive` | Kopya ilişkisinin geçişkenliği (a~b, b~c ⇒ a~c) |
| `test_group_split_keeps_groups_and_ratios` | Bir grup tek bölmede mi, oranlar ±%3 içinde mi, aynı tohumla aynı ayrım mı |
| `test_leakage_audit_catches_duplicate` | Bölmeler arası ortak MD5 olduğunda denetim hata veriyor mu |
| `test_leakage_audit_passes_clean` | Temiz bir ayrımda denetim geçiyor mu (yanlış alarm yok) |
| `test_corruptions_contract` | Her bozulma: boyut/tip korunuyor, deterministik, görüntüyü değiştiriyor, şiddet 5 > şiddet 1 |
| `test_corruption_keeps_labels` | Bozulmuş kopyada etiket dosyası birebir korunuyor mu |

In [ ]:
import tempfile


def _rand_img(seed, h=96, w=128):
    """Düşük frekanslı sentetik renkli görüntü (dHash için anlamlı yapı içerir)."""
    r = np.random.default_rng(seed)
    small = r.integers(0, 256, (12, 16, 3), dtype=np.uint8)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)


def _ham(x, y):
    return int(POPCOUNT[x ^ y].sum())


def test_label_path_for():
    assert label_path_for(Path("root/a/images/train/x.jpg")) == Path("root/a/labels/train/x.txt")
    assert label_path_for(Path("images/sub/images/y.png")) == Path("images/sub/labels/y.txt")


def test_dhash_near_duplicate_and_distinct():
    with tempfile.TemporaryDirectory() as d:
        d = Path(d)
        a, b = _rand_img(1), _rand_img(2)
        imwrite(d / "a.png", a)
        imwrite(d / "a_small.jpg", cv2.resize(a, (64, 48), interpolation=cv2.INTER_AREA))
        imwrite(d / "b.png", b)
        ha, hs, hb = dhash(d / "a.png"), dhash(d / "a_small.jpg"), dhash(d / "b.png")
        assert _ham(ha, hs) <= CFG["dhash_hamming_thr"], f"yakın kopya kaçırıldı: {_ham(ha, hs)} bit"
        assert _ham(ha, hb) > CFG["dhash_hamming_thr"], f"farklı görüntüler kopya sanıldı: {_ham(ha, hb)} bit"


def test_union_find_transitive():
    uf = UnionFind(5)
    uf.union(0, 1)
    uf.union(1, 2)
    assert uf.find(0) == uf.find(2)
    assert uf.find(3) != uf.find(0)


def test_group_split_keeps_groups_and_ratios():
    rng = np.random.default_rng(0)
    gid = [int(g) for g in np.repeat(np.arange(200), rng.integers(1, 6, 200))]
    assign = group_split(gid, CFG["split_ratios"], seed=123)
    splits_of_group = {}
    for i, g in enumerate(gid):
        splits_of_group.setdefault(g, set()).add(assign[i])
    assert all(len(s) == 1 for s in splits_of_group.values()), "bir grup birden fazla bölmeye düştü"
    for s, r in CFG["split_ratios"].items():
        frac = sum(v == s for v in assign.values()) / len(gid)
        assert abs(frac - r) < 0.03, f"{s} oranı {frac:.3f}, hedef {r}"
    assert assign == group_split(gid, CFG["split_ratios"], seed=123), "aynı tohumla farklı ayrım"


def _fake_items(test_md5):
    return [
        {"src": "a", "split": "train", "md5": "m1", "group": 1, "dhash": "ff00ff00ff00ff00"},
        {"src": "b", "split": "val", "md5": "m2", "group": 2, "dhash": "00ff00ff00ff00ff"},
        {"src": "c", "split": "test", "md5": test_md5, "group": 3, "dhash": "0f0f0f0f0f0f0f0f"},
    ]


def test_leakage_audit_catches_duplicate():
    try:
        leakage_audit({"items": _fake_items("m1")}, 5)     # test, train ile aynı MD5'e sahip
    except AssertionError:
        return
    raise AssertionError("sızıntı denetimi ortak MD5'i yakalamadı")


def test_leakage_audit_passes_clean():
    leakage_audit({"items": _fake_items("m3")}, 5)         # hata vermemeli


def test_corruptions_contract():
    img = _rand_img(3)
    for name, fn in CORRUPTIONS.items():
        out1 = fn(img, 3, np.random.default_rng(7))
        out2 = fn(img, 3, np.random.default_rng(7))
        assert out1.shape == img.shape and out1.dtype == np.uint8, f"{name}: boyut/tip değişti"
        assert np.array_equal(out1, out2), f"{name}: deterministik değil"
        assert np.abs(out1.astype(int) - img).mean() > 1.0, f"{name}: görüntüyü değiştirmedi"
        d1 = np.abs(fn(img, 1, np.random.default_rng(7)).astype(int) - img).mean()
        d5 = np.abs(fn(img, 5, np.random.default_rng(7)).astype(int) - img).mean()
        assert d5 > d1, f"{name}: şiddet 5 ({d5:.1f}) şiddet 1'den ({d1:.1f}) az bozuyor"


def test_corruption_keeps_labels():
    with tempfile.TemporaryDirectory() as d:
        d = Path(d)
        src = d / "src"
        (src / "images").mkdir(parents=True)
        (src / "labels").mkdir()
        imwrite(src / "images" / "x.png", _rand_img(4))
        (src / "labels" / "x.txt").write_text("1 0.5 0.5 0.2 0.3\n")
        dst = build_corrupted_split(src, d / "dst", ["fog"], seed=0, fraction=1.0, severity=3)
        assert (dst / "labels" / "x.txt").read_text() == "1 0.5 0.5 0.2 0.3\n"
        assert (dst / "images" / "x.png").exists()


unit_tests = [v for k, v in list(globals().items()) if k.startswith("test_") and callable(v)]
rows = []
for t in unit_tests:
    try:
        t()
        rows.append((t.__name__, "GEÇTİ", ""))
    except Exception as e:
        rows.append((t.__name__, "BAŞARISIZ", f"{type(e).__name__}: {e}"))
unit_report = pd.DataFrame(rows, columns=["test", "sonuç", "ayrıntı"])
unit_report.to_csv(ROOT / "unit_tests.csv", index=False)
display(unit_report)
assert (unit_report["sonuç"] == "GEÇTİ").all(), "Birim testleri başarısız — deneye geçmeden düzeltin."

## 4. Baseline ve ablasyon senaryoları

Her ablasyon, baseline'dan **tek bir faktörü** değiştirir (*ceteris paribus*); böylece fark doğrudan o faktöre atfedilebilir.

| Senaryo | Değişen faktör | Sınadığı hipotez |
|---|---|---|
| `baseline` | — (temiz eğitim, 640 px, varsayılan artırmalar) | Referans nokta |
| `abl_adv_aug` | Eğitim görüntülerinin %50'si bozulmuş (küme boyutu sabit) | Bozulma artırması dayanıklılığı (mPC) artırır mı? Görülmemiş bozulmalara genellenir mi? |
| `abl_imgsz480` | Girdi çözünürlüğü 640 → 480 | Hız–doğruluk eğrisinde hangi model çözünürlük düşüşüne daha duyarlı? |
| `abl_no_mosaic` | Mosaic artırması kapalı | Mosaic'in iki mimariye katkısı farklı mı? |

Toplam çalıştırma = model × senaryo × tohum (`"mid"` profilinde 2 × 3 × 3 = **18**, `"full"` profilinde 2 × 4 × 3 = **24**). Tamamlanan çalıştırmalar diske yazılır ve tekrar çalıştırıldığında atlanır (kesintiye dayanıklı).

In [ ]:
SCENARIOS = {
    "baseline":      {"train": "clean",   "imgsz": 640, "overrides": {}},
    "abl_adv_aug":   {"train": "adverse", "imgsz": 640, "overrides": {}},
    "abl_imgsz480":  {"train": "clean",   "imgsz": 480, "overrides": {}},
    "abl_no_mosaic": {"train": "clean",   "imgsz": 640, "overrides": {"mosaic": 0.0}},
}
ACTIVE_SCENARIOS = P["scenarios"]

YAML_DIR = CFG["data_root"] / "yamls"
YAML_DIR.mkdir(exist_ok=True)


def write_data_yaml(path: Path, train_dir: Path, val_dir: Path, test_dir: Path = None) -> Path:
    lines = [f"path: {CFG['data_root'].as_posix()}",
             f"train: {(train_dir / 'images').as_posix()}",
             f"val: {(val_dir / 'images').as_posix()}"]
    if test_dir is not None:
        lines.append(f"test: {(test_dir / 'images').as_posix()}")
    lines.append("names:")
    lines += [f"  {i}: {json.dumps(n)}" for i, n in enumerate(CFG["class_names"])]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


def train_yaml_for(scn: str, seed: int) -> Path:
    """Eğitim YAML'ı test yolunu BİLEREK içermez: eğitim sırasında test kümesine erişim imkânsızdır."""
    kind = SCENARIOS[scn]["train"]
    if kind == "clean":
        tr = CLEAN / "train"
    else:  # her tohum kendi bozulma örneklemini alır (artırma stokastikliği tohuma dahildir)
        tr = build_corrupted_split(CLEAN / "train", CFG["data_root"] / f"train_adv_s{seed}",
                                   CFG["train_corruptions"], seed=seed, fraction=CFG["adv_fraction"])
    return write_data_yaml(YAML_DIR / f"train_{kind}_s{seed}.yaml", tr, CLEAN / "val")


EVAL_YAMLS = {c: write_data_yaml(YAML_DIR / f"eval_{c}.yaml", CLEAN / "train", CLEAN / "val", d)
              for c, d in TEST_DIRS.items()}
n_runs = len(CFG["models"]) * len(ACTIVE_SCENARIOS) * len(CFG["seeds"])
print(f"Aktif senaryolar: {ACTIVE_SCENARIOS}  |  toplam çalıştırma: {n_runs}")

## 5. Eğitim, değerlendirme ve hız ölçümü

Her çalıştırma için:

1. `set_seed(seed)` + Ultralytics `seed=..., deterministic=True`.
2. Eğitim; **en iyi ağırlık (`best.pt`) yalnızca val mAP'ine göre** seçilir.
3. `best.pt` ile 6 test koşulunda (temiz + 5 bozulma) mAP@[.5:.95] ve mAP@.5.
4. Dayanıklılık özetleri: **mPC** = bozulmalar üzerinden ortalama mAP, **rPC** = mPC / temiz mAP.
5. **FPS (uçtan uca, batch=1):** ön-işleme + çıkarım + son-işleme (YOLO için NMS dahil; DETR NMS'sizdir — bu farkın kendisi karşılaştırmanın parçasıdır). Isınma turları atılır, her ölçümde `torch.cuda.synchronize()` çağrılır; FPS medyan gecikmeden hesaplanır (aykırı değerlere dayanıklı).

> FPS donanıma bağlıdır; aynı oturumda, aynı GPU'da, arka planda başka iş yokken ölçün. Ortam bilgisi `environment.json`'a yazılır.

In [ ]:
METRICS_DIR = CFG["runs_root"] / "_metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)


def load_model(family: str, weights):
    return (RTDETR if family == "rtdetr" else YOLO)(str(weights))


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


@torch.no_grad()
def measure_fps(model, img_dir: Path, imgsz: int):
    imgs = [imread(p) for p in sorted(img_dir.iterdir())[:32]]
    kw = dict(imgsz=imgsz, device=CFG["device"], conf=0.25, half=False, verbose=False)
    for i in range(CFG["fps_warmup"]):
        model.predict(imgs[i % len(imgs)], **kw)
    lat = []
    for i in range(CFG["fps_iters"]):
        cuda_sync()
        t0 = time.perf_counter()
        model.predict(imgs[i % len(imgs)], **kw)
        cuda_sync()
        lat.append((time.perf_counter() - t0) * 1000.0)
    lat = np.asarray(lat)
    return {"latency_ms": float(np.median(lat)), "latency_p95_ms": float(np.percentile(lat, 95)),
            "fps": float(1000.0 / np.median(lat))}


def run_experiment(model_key: str, scn: str, seed: int) -> dict:
    family, weights = CFG["models"][model_key]
    sc = SCENARIOS[scn]
    name = f"{model_key}__{scn}__s{seed}__{PROFILE}"   # profil eki: farklı profillerin sonuçları karışmaz
    out_json = METRICS_DIR / f"{name}.json"
    if out_json.exists():
        print(f"✓ atlanıyor (tamamlanmış): {name}")
        return json.loads(out_json.read_text(encoding="utf-8"))

    print(f"\n▶ {name}")
    set_seed(seed)
    model = load_model(family, weights)
    model.train(data=str(train_yaml_for(scn, seed)), epochs=CFG["epochs"], imgsz=sc["imgsz"],
                batch=CFG["batch"], fraction=CFG["train_fraction"], seed=seed, deterministic=True,
                device=CFG["device"], workers=CFG["workers"], project=str(CFG["runs_root"]), name=name,
                exist_ok=True, plots=False, verbose=False, **sc["overrides"])
    save_dir = Path(model.trainer.save_dir)

    model = load_model(family, save_dir / "weights" / "best.pt")
    row = {"model": model_key, "family": family, "scenario": scn, "seed": seed, "imgsz": sc["imgsz"],
           "save_dir": str(save_dir),
           "params_M": sum(p.numel() for p in model.model.parameters()) / 1e6}

    for cond, y in EVAL_YAMLS.items():        # test kümesine İLK ve TEK erişim burada
        r = model.val(data=str(y), split="test", imgsz=sc["imgsz"], batch=CFG["batch"], device=CFG["device"],
                      project=str(save_dir), name=f"test_{cond}", exist_ok=True, plots=False, verbose=False)
        row[f"mAP50_95_{cond}"] = float(r.box.map)
        row[f"mAP50_{cond}"] = float(r.box.map50)

    corr = [row[f"mAP50_95_{c}"] for c in CFG["test_corruptions"]]
    row["mPC"] = float(np.mean(corr))
    row["rPC"] = row["mPC"] / max(row["mAP50_95_clean"], 1e-9)
    row.update(measure_fps(model, TEST_DIRS["clean"] / "images", sc["imgsz"]))

    out_json.write_text(json.dumps(row, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

In [ ]:
# Ana döngü — kesilirse yeniden çalıştırın; tamamlananlar atlanır
t_start = time.time()
rows = []
for m in CFG["models"]:
    for scn in ACTIVE_SCENARIOS:
        for seed in CFG["seeds"]:
            rows.append(run_experiment(m, scn, seed))

results = pd.DataFrame(rows)
results.to_csv(ROOT / "results_per_run.csv", index=False)
print(f"\nBitti: {len(results)} çalıştırma, {(time.time() - t_start) / 60:.1f} dk")
results[["model", "scenario", "seed", "mAP50_95_clean", "mPC", "rPC", "fps"]].round(4)

## 6. İstatistiksel özet: ortalama ± standart sapma

* **Std**: örneklem standart sapması (`ddof=1`), n = 3 tohum.
* **%95 GA**: Student-t ile, $\bar{x} \pm t_{0.975,\,n-1}\, s/\sqrt{n}$ (n=3 için $t \approx 4.30$ — aralıklar geniştir, bu beklenen bir durumdur).
* **Eşleştirilmiş t-testi**: aynı tohum aynı veri sırası/bozulma örneklemini paylaştığı için karşılaştırmalar tohum bazında eşleştirilir. Çoklu karşılaştırma için **Holm–Bonferroni** düzeltmesi uygulanır.
* n=3 ile testin gücü düşüktür: *anlamlı değil* ≠ *fark yok*. Etki büyüklüğünü (Δ ort. ± std) p-değerinden önce okuyun.

In [ ]:
METRICS = (["mAP50_95_clean", "mAP50_clean", "mPC", "rPC", "fps", "latency_ms", "params_M"]
           + [f"mAP50_95_{c}" for c in CFG["test_corruptions"]])


def summarise(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for (m, s), g in df.groupby(["model", "scenario"], sort=False):
        rec = {"model": m, "scenario": s, "n_seeds": len(g)}
        for k in METRICS:
            x = g[k].to_numpy(float)
            n = len(x)
            sd = x.std(ddof=1) if n > 1 else np.nan
            rec[f"{k}_mean"] = x.mean()
            rec[f"{k}_std"] = sd
            rec[f"{k}_ci95"] = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan
        out.append(rec)
    return pd.DataFrame(out)


def pm(df, k, fmt="{:.3f}"):
    return df[f"{k}_mean"].map(fmt.format) + " ± " + df[f"{k}_std"].map(fmt.format)


summary = summarise(results)
summary.to_csv(ROOT / "results_summary.csv", index=False)

table = summary[["model", "scenario", "n_seeds"]].copy()
table["Param (M)"] = summary["params_M_mean"].map("{:.1f}".format)
table["mAP@[.5:.95] temiz"] = pm(summary, "mAP50_95_clean")
table["mAP@.5 temiz"] = pm(summary, "mAP50_clean")
table["mPC"] = pm(summary, "mPC")
table["rPC"] = pm(summary, "rPC")
table["FPS"] = pm(summary, "fps", "{:.1f}")
table["Gecikme (ms)"] = pm(summary, "latency_ms", "{:.1f}")
print("Ana tablo (ortalama ± std, 3 tohum)")
display(table)

per_corr = summary[["model", "scenario"]].copy()
for c in ["clean"] + CFG["test_corruptions"]:
    per_corr[c] = pm(summary, f"mAP50_95_{c}")
print("Koşul başına mAP@[.5:.95] (ortalama ± std)")
display(per_corr)

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(np.where(np.isnan(p), np.inf, p))
    adj = np.full(m, np.nan)
    running = 0.0
    for rank, i in enumerate(order):
        if np.isnan(p[i]):
            continue
        running = max(running, (m - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


def paired(a_df, b_df, metric):
    a = a_df.sort_values("seed")[metric].to_numpy(float)
    b = b_df.sort_values("seed")[metric].to_numpy(float)
    d = a - b
    t, p = stats.ttest_rel(a, b)
    return d.mean(), d.std(ddof=1), t, p


sel = lambda m, s: results[(results["model"] == m) & (results["scenario"] == s)]
model_keys = list(CFG["models"])
A, B = model_keys[0], model_keys[1]
tests = []

# (a) Mimari karşılaştırması: aynı senaryo, aynı tohumlar
for scn in ACTIVE_SCENARIOS:
    for met in ["mAP50_95_clean", "mPC", "fps"]:
        dm, ds, t, p = paired(sel(A, scn), sel(B, scn), met)
        tests.append({"karşılaştırma": f"{A} − {B}", "senaryo": scn, "metrik": met, "Δ ort": dm, "Δ std": ds, "t": t, "p": p})

# (b) Ablasyon etkisi: her model için ablasyon − baseline
for m in model_keys:
    for scn in ACTIVE_SCENARIOS:
        if scn == "baseline":
            continue
        for met in ["mAP50_95_clean", "mPC", "fps"]:
            dm, ds, t, p = paired(sel(m, scn), sel(m, "baseline"), met)
            tests.append({"karşılaştırma": f"{m}: {scn} − baseline", "senaryo": scn, "metrik": met, "Δ ort": dm, "Δ std": ds, "t": t, "p": p})

tests_df = pd.DataFrame(tests)
tests_df["p_holm"] = holm(tests_df["p"])
tests_df["anlamlı (α=.05, Holm)"] = tests_df["p_holm"] < 0.05
tests_df.to_csv(ROOT / "stat_tests.csv", index=False)
display(tests_df.round(4))

## 7. Grafikler

1. **Eğitim eğrileri** — eğitim kaybı, doğrulama kaybı ve doğrulama mAP'i; çizgi = tohum ortalaması, gölge = ±1 std.
   ⚠️ YOLO (box + cls + DFL) ve RT-DETR (GIoU + L1 + sınıflandırma) **farklı kayıp fonksiyonları** kullanır; mutlak kayıp değerleri modeller arasında **karşılaştırılamaz**. Bu yüzden her model ayrı satırdadır; karşılaştırılabilir ortak ölçü mAP'tir.
2. **mAP–FPS ödünleşimi** — hata çubukları her iki eksende ±1 std; kesikli çizgi Pareto sınırı (daha hızlı *ve* daha doğru başka nokta yok).
3. **Koşul başına dayanıklılık** — † = eğitimde (ablasyonda bile) hiç görülmeyen bozulma.

In [ ]:
# Ortak stil: sabit sıralı kategorik palet (renk varlığı izler, sıralamayı değil), sade ızgara
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
MODEL_COLOR = {m: PALETTE[i] for i, m in enumerate(CFG["models"])}
MODEL_MARKER = {m: mk for m, mk in zip(CFG["models"], ["o", "s", "^", "D"])}
SCN_COLOR = {s: PALETTE[i] for i, s in enumerate(SCENARIOS)}   # tüm senaryolara sabit atanır -> filtre renkleri değiştirmez

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "font.size": 10, "figure.dpi": 110,
})


def load_curves(save_dir) -> pd.DataFrame:
    df = pd.read_csv(Path(save_dir) / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    tr = [c for c in df.columns if c.startswith("train/") and "loss" in c]   # YOLO: box/cls/dfl, RT-DETR: giou/cls/l1
    va = [c for c in df.columns if c.startswith("val/") and "loss" in c]
    return pd.DataFrame({"epoch": df["epoch"].astype(int), "train_loss": df[tr].sum(axis=1),
                         "val_loss": df[va].sum(axis=1), "mAP50_95": df["metrics/mAP50-95(B)"],
                         "mAP50": df["metrics/mAP50(B)"]})


curves = []
for _, r in results.iterrows():
    c = load_curves(r["save_dir"])
    c["model"], c["scenario"], c["seed"] = r["model"], r["scenario"], r["seed"]
    curves.append(c)
curves = pd.concat(curves, ignore_index=True)

In [ ]:
# Grafik 1 — Kayıp ve mAP eğrileri (ortalama ± 1 std, 3 tohum)
panels = [("train_loss", "Eğitim kaybı (bileşenler toplamı)"),
          ("val_loss", "Doğrulama kaybı (bileşenler toplamı)"),
          ("mAP50_95", "Doğrulama mAP@[.5:.95]")]
fig, axes = plt.subplots(len(model_keys), 3, figsize=(15, 3.8 * len(model_keys)), squeeze=False)
for i, m in enumerate(model_keys):
    for j, (col, title) in enumerate(panels):
        ax = axes[i, j]
        for scn in ACTIVE_SCENARIOS:
            g = (curves[(curves["model"] == m) & (curves["scenario"] == scn)]
                 .groupby("epoch")[col].agg(["mean", "std"]).reset_index())
            sd = g["std"].fillna(0)
            ax.plot(g["epoch"], g["mean"], color=SCN_COLOR[scn], label=scn)
            ax.fill_between(g["epoch"], g["mean"] - sd, g["mean"] + sd, color=SCN_COLOR[scn], alpha=0.18, linewidth=0)
        ax.set_title(f"{m} — {title}", fontsize=10, loc="left")
        ax.set_xlabel("Epoch")
axes[0, 0].legend(title="Senaryo", frameon=False)
fig.suptitle("Eğitim eğrileri (çizgi: tohum ortalaması, gölge: ±1 std, n=3)", x=0.01, ha="left")
plt.tight_layout()
fig.savefig(CFG["fig_dir"] / "1_training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Grafik 2 — mAP–FPS ödünleşimi ve Pareto sınırı
def pareto_front(pts):
    """(fps, mAP) noktaları; daha yüksek FPS ve daha yüksek mAP iyidir."""
    front, best = [], -np.inf
    for x, y in sorted(pts, key=lambda p: -p[0]):
        if y > best:
            front.append((x, y))
            best = y
    return sorted(front)


fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
for ax, (k, title) in zip(axes, [("mAP50_95_clean", "Temiz test"),
                                 ("mPC", f"Zorlu koşullar — mPC (şiddet {CFG['test_severity']})")]):
    pts = []
    for _, r in summary.iterrows():
        col = SCN_COLOR[r["scenario"]]
        ax.errorbar(r["fps_mean"], r[f"{k}_mean"], xerr=r["fps_std"], yerr=r[f"{k}_std"],
                    fmt=MODEL_MARKER[r["model"]], ms=9, color=col, mec=SURFACE, mew=1.5,
                    ecolor=col, elinewidth=1.2, capsize=3, zorder=3)
        pts.append((r["fps_mean"], r[f"{k}_mean"]))
    fx, fy = zip(*pareto_front(pts))
    ax.plot(fx, fy, ls="--", lw=1.2, color="#8a8984", zorder=1)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel("FPS (uçtan uca, batch=1)  →  daha hızlı")
    ax.set_ylabel("mAP@[.5:.95]  →  daha doğru")

handles = ([Line2D([], [], ls="", marker=MODEL_MARKER[m], ms=9, color=INK2, label=m) for m in model_keys]
           + [Patch(color=SCN_COLOR[s], label=s) for s in ACTIVE_SCENARIOS]
           + [Line2D([], [], ls="--", color="#8a8984", label="Pareto sınırı")])
axes[1].legend(handles=handles, frameon=False, loc="best")
fig.suptitle("Hız–doğruluk ödünleşimi (ortalama ± 1 std, n=3 tohum)", x=0.01, ha="left")
plt.tight_layout()
fig.savefig(CFG["fig_dir"] / "2_map_fps_tradeoff.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Grafik 3 — Koşul başına dayanıklılık (baseline vs. bozulma artırmalı eğitim)
conds = ["clean"] + CFG["test_corruptions"]
unseen = set(CFG["test_corruptions"]) - set(CFG["train_corruptions"])
show_scn = [s for s in ("baseline", "abl_adv_aug") if s in ACTIVE_SCENARIOS]
combos = [(m, s) for m in model_keys for s in show_scn]

x = np.arange(len(conds))
w = 0.8 / len(combos)
fig, ax = plt.subplots(figsize=(13, 5))
for k, (m, s) in enumerate(combos):
    r = summary[(summary["model"] == m) & (summary["scenario"] == s)].iloc[0]
    ax.bar(x - 0.4 + w * (k + 0.5),
           [r[f"mAP50_95_{c}_mean"] for c in conds], w * 0.9,
           yerr=[r[f"mAP50_95_{c}_std"] for c in conds],
           color=MODEL_COLOR[m], hatch="//" if s != "baseline" else None, edgecolor=SURFACE,
           linewidth=0, capsize=2, error_kw={"elinewidth": 1, "ecolor": INK2}, label=f"{m} · {s}")
ax.set_xticks(x)
ax.set_xticklabels([c + (" †" if c in unseen else "") for c in conds])
ax.set_ylabel("Test mAP@[.5:.95]")
ax.set_xlabel("Test koşulu   († = eğitimde hiç görülmeyen bozulma)")
ax.legend(frameon=False, ncol=len(combos))
ax.set_title("Koşul başına dayanıklılık (ortalama ± 1 std, n=3 tohum; taralı = bozulma artırmalı eğitim)",
             loc="left", fontsize=11)
plt.tight_layout()
fig.savefig(CFG["fig_dir"] / "3_robustness_per_condition.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Yorumlama rehberi ve sınırlılıklar

**Okuma sırası önerisi**
1. *Sızıntı denetimi* tablosu geçti mi? (Geçmediyse hiçbir sonuç geçerli değildir.)
2. *Ana tablo*: temiz mAP ve FPS → hangi model Pareto sınırında?
3. *mPC / rPC*: zorlu koşullarda hangi mimari **göreli olarak** daha az kayıp yaşıyor? (Transformer'ların küresel dikkat mekanizmasının yerel doku bozulmalarına — gürültü, yağmur — daha dayanıklı olduğu hipotezi burada sınanır.)
4. *Ablasyon Δ'ları*: bozulma artırması görülen (`fog`, `low_light`, `gaussian_noise`) ve görülmeyen (`motion_blur` †, `rain` †) bozulmalarda aynı ölçüde mi yardımcı oluyor?
5. Eğitim eğrileri: aşırı öğrenme (val kaybı yükselirken eğitim kaybı düşüyor) veya erken doygunluk var mı? RT-DETR genelde daha yavaş yakınsar — epoch bütçesi bir modeli kayırıyor olabilir.

**Sınırlılıklar (raporda belirtilmeli)**
* **n = 3 tohum** küçük bir örneklemdir; std tahmininin kendisi de belirsizdir ve t-testinin gücü düşüktür. Mümkünse 5 tohuma çıkın.
* Tohumlar yalnızca **eğitim** stokastikliğini kapsar; **veri ayrımı** varyansı sabittir. Ayrım varyansını da ölçmek için farklı `split_seed`'lerle (iç içe tasarım) tekrarlayın.
* Bozulmalar **sentetiktir**; gerçek dünya geçerliliği için ExDark (düşük ışık), DAWN / RTTS (sis, yağmur) gibi gerçek zorlu koşul kümeleri ile doğrulayın (`source_dir` ile kendi kümenizi verebilirsiniz).
* FPS ölçümü PyTorch FP32, batch=1 içindir; TensorRT/FP16 dağıtımında sıralama değişebilir. YOLO'nun NMS süresi sahnedeki tespit sayısına bağlıdır.
* Her model Ultralytics'in **kendi varsayılan eğitim reçetesiyle** eğitilir; mimari başına hiperparametre ayarı yapılmamıştır (adil ama ikisi için de optimal olmayabilir).
* Test kümesi yalnızca en sonda kullanıldı; bu sonuçlara bakarak senaryo/hiperparametre değiştirirseniz, test kümesi artık *görülmüş* sayılır — yeni bir ayrım gerekir.

**Üretilen dosyalar** (`cv_bench/`): `results_per_run.csv`, `results_summary.csv`, `stat_tests.csv`, `environment.json`, `data/split_manifest.json`, `figures/*.png`.